# Nonparametric estimation: two-way fixed-effects kernel regression

Local linear regression of the doubly demeaned Euler residual $\tilde y_{it} = y_{it} - \hat\alpha_i - \hat\tau_t$ on $(\text{lev}, \sigma)$ for banks and non-banks. Bootstrap-based CIs from a fast cluster bootstrap. Recovers $\hat f$ by integrating $\hat g$ along leverage, anchored at the median.

## Cell 1: Imports and load data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
import time
import warnings
warnings.filterwarnings('ignore')

PROD_DATA_PATH = '/kaggle/input/datasets/lucanadig/metrics-project/panel_data.csv'
df_full = pd.read_csv(PROD_DATA_PATH)
df_full['date'] = pd.to_datetime(df_full['date'])
print(f'Loaded {len(df_full)} rows, {df_full["gvkey"].nunique()} firms')
print(f'Bank classification:\n{df_full["bank"].value_counts()}')

## Cell 2: Prepare bank and non-bank subsets

In [ ]:
def prepare_subset(df, classification, lev_quantile_cap=0.99,
                    y_quantile_trim=0.01, port_std_quantile_cap=0.99):
    sub = df[df['bank'] == classification].copy()
    
    # New leverage definition: assets = atq, equity = ceqq
    sub['equity'] = sub['ceqq']
    
    sub = sub[(sub['equity'] > 0) & (sub['port_std'] > 0)].copy()
    sub['leverage'] = sub['assets'] / sub['equity']
    sub = sub.replace([np.inf, -np.inf], np.nan).dropna(
        subset=['leverage', 'port_std', 'euler_equation', 'gvkey']
    )
    n0 = len(sub)
    sub = sub[sub['leverage'] <= sub['leverage'].quantile(lev_quantile_cap)].copy()
    sub = sub[sub['port_std'] <= sub['port_std'].quantile(port_std_quantile_cap)].copy()
    y_lo, y_hi = sub['euler_equation'].quantile([y_quantile_trim, 1 - y_quantile_trim])
    sub = sub[sub['euler_equation'].between(y_lo, y_hi)].copy()
    sub = sub.sort_values(['gvkey', 'date']).reset_index(drop=True)
    
    print(f'{classification}: dropped {n0 - len(sub)} of {n0} obs')
    print(f'  Final: {len(sub)} obs, {sub["gvkey"].nunique()} firms')
    print(f'  Leverage range: [{sub["leverage"].min():.2f}, {sub["leverage"].max():.2f}]')
    print(f'  port_std range: [{sub["port_std"].min():.4f}, {sub["port_std"].max():.4f}]')
    return sub

df_bank = prepare_subset(df_full, 'Bank')
print()
df_nonbank = prepare_subset(df_full, 'Non-Bank')

## Cell 3: Demeaning function for fixed effects

In [ ]:
def apply_demeaning(sub, fe='none'):
    """
    Return y after applying chosen FE demeaning.
    
    fe: 'none', 'bank', 'time', or 'two_way'
    """
    y = sub['euler_equation'].values.copy()
    if fe == 'none':
        return y
    if fe == 'bank':
        return y - sub.groupby('gvkey')['euler_equation'].transform('mean').values
    if fe == 'time':
        return y - sub.groupby('date')['euler_equation'].transform('mean').values
    if fe == 'two_way':
        # Iterative demeaning (works for unbalanced panels)
        ydm = y.copy()
        for _ in range(15):
            df_tmp = pd.DataFrame({
                'y': ydm,
                'gvkey': sub['gvkey'].values,
                'date': sub['date'].values,
            })
            ydm = ydm - df_tmp.groupby('gvkey')['y'].transform('mean').values
            df_tmp['y'] = ydm
            ydm = ydm - df_tmp.groupby('date')['y'].transform('mean').values
        return ydm
    raise ValueError(f"unknown fe: {fe}")

## Cell 4: Local linear mean regression with interior masking

In [ ]:
def fit_g_nonparametric(
    x1, x2, y,
    grid_x1=None, grid_x2=None,
    resolution=20,
    n_neighbors=1000,
    bandwidth_factor=2.0,
    grid_method='quantile',
    interior_distance_quantile=0.75,
    interior_safety=1.5,
):
    s1, s2 = np.std(x1), np.std(x2)
    pts = np.column_stack([x1 / s1, x2 / s2])
    tree = cKDTree(pts)

    if grid_x1 is None:
        grid_x1 = (np.quantile(x1, np.linspace(0.02, 0.98, resolution))
                   if grid_method == 'quantile'
                   else np.linspace(x1.min(), x1.max(), resolution))
    if grid_x2 is None:
        grid_x2 = (np.quantile(x2, np.linspace(0.02, 0.98, resolution))
                   if grid_method == 'quantile'
                   else np.linspace(x2.min(), x2.max(), resolution))

    X1_g, X2_g = np.meshgrid(grid_x1, grid_x2)
    grid_pts = np.column_stack([X1_g.ravel() / s1, X2_g.ravel() / s2])
    n_query = len(grid_pts)
    n_neighbors = min(n_neighbors, len(x1))
    dists, idx = tree.query(grid_pts, k=n_neighbors)

    flat_g = np.full(n_query, np.nan)
    for q in range(n_query):
        nbr = idx[q]; d = dists[q]
        h = d.max() * bandwidth_factor
        if h <= 0: continue
        u = d / h
        w = np.where(u < 1, (1 - u**3)**3, 0)
        if w.sum() < 5: continue
        gp1, gp2 = grid_pts[q]
        x1_n = pts[nbr, 0] - gp1; x2_n = pts[nbr, 1] - gp2
        y_n = y[nbr]
        X = np.column_stack([np.ones(len(nbr)), x1_n, x2_n])
        sw = np.sqrt(w)
        try:
            coef, *_ = np.linalg.lstsq(X * sw[:, None], y_n * sw, rcond=None)
            flat_g[q] = coef[0]
        except np.linalg.LinAlgError: pass

    G_grid = flat_g.reshape(X1_g.shape)
    median_dist = np.median(dists, axis=1).reshape(X1_g.shape)
    thresh = np.quantile(median_dist, interior_distance_quantile) * interior_safety
    support_mask = median_dist <= thresh
    return X1_g, X2_g, G_grid, support_mask

## Cell 5: Point-estimate fits (sanity check)

Quick fits to check the pipeline runs and the data look reasonable, before doing the expensive bootstrap. Uses `fe='two_way'` to match the headline specification.

In [ ]:
FIT_KWARGS = dict(
    resolution=20,
    n_neighbors=1000,
    bandwidth_factor=2.0,
    grid_method='quantile',
)

def run_fit(sub_df, label, fe='two_way'):
    x1 = sub_df['leverage'].values
    x2 = sub_df['port_std'].values
    y  = apply_demeaning(sub_df, fe=fe)
    t0 = time.time()
    X1, X2, G, M = fit_g_nonparametric(x1, x2, y, **FIT_KWARGS)
    print(f'{label} (fe={fe}): {time.time()-t0:.1f}s, '
          f'{int(M.sum())}/{M.size} cells in support')
    return {'X1': X1, 'X2': X2, 'G': G, 'mask': M,
            'x1': x1, 'x2': x2, 'y': y, 'label': label, 'fe': fe}

fit_b = run_fit(df_bank, 'Bank')
fit_n = run_fit(df_nonbank, 'Non-Bank')

## Cell 6: $\hat g$ heatmaps (sanity check)

In [ ]:
def plot_heatmap(fit, ax, vlim):
    G = np.where(fit['mask'], fit['G'], np.nan)
    pcm = ax.pcolormesh(fit['X1'], fit['X2'], G, shading='auto', cmap='RdBu_r',
                        vmin=-vlim, vmax=vlim)
    rng = np.random.default_rng(0)
    n_show = min(2000, len(fit['x1']))
    show_idx = rng.choice(len(fit['x1']), size=n_show, replace=False)
    ax.scatter(fit['x1'][show_idx], fit['x2'][show_idx], s=2, alpha=0.2, color='black')
    ax.set_xlabel('Leverage  a/e')
    ax.set_ylabel('port_std')
    ax.set_title(fit['label'])
    return pcm

vlim = max(
    np.nanpercentile(np.abs(fit_b['G'][fit_b['mask']]), 95),
    np.nanpercentile(np.abs(fit_n['G'][fit_n['mask']]), 95),
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, fit in zip(axes, [fit_b, fit_n]):
    pcm = plot_heatmap(fit, ax, vlim)
fig.colorbar(pcm, ax=axes, label=r'$\hat g$', shrink=0.8)
fig.suptitle(r'Local linear $\hat g(\text{lev}, \sigma)$')
plt.show()

## Cell 7: Fast cluster bootstrap

Precomputes KDTree + neighbor lists + kernel weights once on the full sample. Each bootstrap rep multiplies kernel weights by firm multiplicity in the resample — no rebuild, no re-query. Parallelized across reps with `joblib`.

In [ ]:
def precompute_kernel_structure(x1, x2, fit_kwargs):
    """Compute KDTree + neighbor lists + kernel weights ONCE for the full sample."""
    s1, s2 = np.std(x1), np.std(x2)
    pts = np.column_stack([x1 / s1, x2 / s2])
    tree = cKDTree(pts)

    res = fit_kwargs['resolution']
    method = fit_kwargs.get('grid_method', 'quantile')
    if method == 'quantile':
        grid_x1 = np.quantile(x1, np.linspace(0.02, 0.98, res))
        grid_x2 = np.quantile(x2, np.linspace(0.02, 0.98, res))
    else:
        grid_x1 = np.linspace(x1.min(), x1.max(), res)
        grid_x2 = np.linspace(x2.min(), x2.max(), res)
    X1_g, X2_g = np.meshgrid(grid_x1, grid_x2)
    grid_pts = np.column_stack([X1_g.ravel() / s1, X2_g.ravel() / s2])

    n_neighbors = min(fit_kwargs['n_neighbors'], len(x1))
    dists, idx = tree.query(grid_pts, k=n_neighbors)

    bf = fit_kwargs.get('bandwidth_factor', 2.0)
    h = dists.max(axis=1) * bf
    h_safe = np.where(h > 0, h, 1.0)
    u = dists / h_safe[:, None]
    kernel_w = np.where(u < 1, (1 - u**3)**3, 0)

    x_n_1 = pts[idx, 0] - grid_pts[:, 0:1]
    x_n_2 = pts[idx, 1] - grid_pts[:, 1:2]

    median_dist = np.median(dists, axis=1).reshape(X1_g.shape)
    iq = fit_kwargs.get('interior_distance_quantile', 0.75)
    isf = fit_kwargs.get('interior_safety', 1.5)
    thresh = np.quantile(median_dist, iq) * isf
    mask = median_dist <= thresh

    return {
        'X1': X1_g, 'X2': X2_g, 'grid_pts': grid_pts,
        'idx': idx, 'kernel_w': kernel_w,
        'x_n_1': x_n_1, 'x_n_2': x_n_2,
        'mask': mask,
    }


def fit_g_at_grid(struct, y, weights):
    """Weighted local linear MEAN regression at every grid point."""
    idx = struct['idx']
    kernel_w = struct['kernel_w']
    x_n_1 = struct['x_n_1']
    x_n_2 = struct['x_n_2']
    n_grid, k = idx.shape

    w_full = kernel_w * weights[idx]   # (n_grid, k)

    flat_g = np.full(n_grid, np.nan)
    for g in range(n_grid):
        w = w_full[g]
        if w.sum() < 5: continue
        sw = np.sqrt(w)
        nbr = idx[g]
        X = np.column_stack([np.ones(k), x_n_1[g], x_n_2[g]])
        try:
            coef, *_ = np.linalg.lstsq(X * sw[:, None], y[nbr] * sw, rcond=None)
            flat_g[g] = coef[0]
        except np.linalg.LinAlgError:
            pass
    return flat_g.reshape(struct['X1'].shape)


def _one_boot_rep(struct, y, gvkey_to_obs, unique_keys, n_obs, seed):
    rng = np.random.default_rng(seed)
    sampled = rng.choice(unique_keys, size=len(unique_keys), replace=True)
    weights = np.zeros(n_obs)
    for k in sampled:
        weights[gvkey_to_obs[k]] += 1.0
    return fit_g_at_grid(struct, y, weights)


def fast_cluster_bootstrap(sub_df, B=100, seed=0, fit_kwargs=None,
                            n_jobs=-1, fe='none'):
    if fit_kwargs is None: fit_kwargs = FIT_KWARGS
    x1 = sub_df['leverage'].values
    x2 = sub_df['port_std'].values
    y  = apply_demeaning(sub_df, fe=fe)
    gvkey = sub_df['gvkey'].values
    n_obs = len(y)
    unique_keys = np.unique(gvkey)
    gvkey_to_obs = {k: np.where(gvkey == k)[0] for k in unique_keys}

    print(f'  Precomputing kernel structure ({n_obs} obs, '
          f'grid={fit_kwargs["resolution"]}^2, fe={fe})...')
    t0 = time.time()
    struct = precompute_kernel_structure(x1, x2, fit_kwargs)
    print(f'    done in {time.time()-t0:.1f}s')

    print(f'  Original fit...')
    t0 = time.time()
    weights_orig = np.ones(n_obs)
    G_orig = fit_g_at_grid(struct, y, weights_orig)
    print(f'    done in {time.time()-t0:.1f}s')

    print(f'  Running {B} bootstrap reps with n_jobs={n_jobs}...')
    t0 = time.time()
    seeds = [seed * 100000 + b for b in range(B)]
    boot_results = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(_one_boot_rep)(struct, y, gvkey_to_obs, unique_keys, n_obs, s)
        for s in seeds
    )
    print(f'    done in {time.time()-t0:.1f}s')

    G_boot = np.stack(boot_results, axis=0)
    return struct, G_orig, G_boot

## Cell 8: Run bootstrap with two-way fixed effects

In [ ]:
B_BOOTSTRAP = 100
PRIMARY_FE = 'two_way'

print(f'Bootstrap with fe={PRIMARY_FE}: banks')
struct_b, G_orig_b, G_boot_b = fast_cluster_bootstrap(
    df_bank, B=B_BOOTSTRAP, seed=0, fit_kwargs=FIT_KWARGS, fe=PRIMARY_FE,
)
print(f'\nBootstrap with fe={PRIMARY_FE}: non-banks')
struct_n, G_orig_n, G_boot_n = fast_cluster_bootstrap(
    df_nonbank, B=B_BOOTSTRAP, seed=0, fit_kwargs=FIT_KWARGS, fe=PRIMARY_FE,
)

## Cell 9: Plotting functions for $\hat g$ slices with bootstrap CIs

In [ ]:
def plot_three_slices_with_ci(struct, G_orig, G_boot, port_std_data,
                                quantiles=(0.25, 0.5, 0.75), label='', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(quantiles)))
    targets = np.quantile(port_std_data, quantiles)

    for q, target, color in zip(quantiles, targets, colors):
        j = np.argmin(np.abs(struct['X2'][:, 0] - target))
        x_axis = struct['X1'][j]
        mask_row = struct['mask'][j]
        base = np.where(mask_row, G_orig[j], np.nan)
        boot = np.where(mask_row[None, :], G_boot[:, j, :], np.nan)
        lo = np.nanpercentile(boot, 2.5, axis=0)
        hi = np.nanpercentile(boot, 97.5, axis=0)
        ax.fill_between(x_axis, lo, hi, alpha=0.18, color=color)
        ax.plot(x_axis, base, color=color, lw=2,
                label=f'port_std = {target:.3f} ({int(q*100)}th)')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xlabel('Leverage  a/e')
    ax.set_ylabel(r'$\hat g$')
    ax.set_title(f'{label}: $\\hat g$ vs leverage')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

## Cell 10: Recover $\hat f$ by integrating $\hat g$

Anchored at the median leverage in each subset, so CIs widen symmetrically on either side of the anchor rather than only outward from a boundary.

In [ ]:
def recover_f_from_g(X1, X2, G, mask, anchor_lev=None):
    """Cumulative trapezoidal integration of g along leverage."""
    G_clean = np.where(mask, G, np.nan)
    lev_axis = X1[0]
    if anchor_lev is None:
        anchor_lev = lev_axis[0]
    F = np.full_like(G_clean, np.nan)
    i_anchor = np.argmin(np.abs(lev_axis - anchor_lev))

    for j in range(G_clean.shape[0]):
        row = G_clean[j]
        if not np.any(~np.isnan(row)):
            continue
        F_row = np.full_like(row, np.nan)
        F_row[i_anchor] = 0.0
        for i in range(i_anchor + 1, len(row)):
            if np.isnan(row[i]) or np.isnan(row[i-1]):
                F_row[i] = np.nan
            else:
                F_row[i] = F_row[i-1] + 0.5*(row[i]+row[i-1])*(lev_axis[i]-lev_axis[i-1])
        for i in range(i_anchor - 1, -1, -1):
            if np.isnan(row[i]) or np.isnan(row[i+1]):
                F_row[i] = np.nan
            else:
                F_row[i] = F_row[i+1] - 0.5*(row[i]+row[i+1])*(lev_axis[i+1]-lev_axis[i])
        F[j] = F_row
    return F


def recover_f_with_ci(struct, G_orig, G_boot, anchor_lev=None):
    F_base = recover_f_from_g(struct['X1'], struct['X2'],
                                G_orig, struct['mask'],
                                anchor_lev=anchor_lev)
    F_boot = np.full_like(G_boot, np.nan)
    for b in range(G_boot.shape[0]):
        F_boot[b] = recover_f_from_g(struct['X1'], struct['X2'],
                                      G_boot[b], struct['mask'],
                                      anchor_lev=anchor_lev)
    return F_base, F_boot


# Anchor at the MEDIAN leverage in each subset
ANCHOR_LEV_B = float(np.median(df_bank['leverage']))
ANCHOR_LEV_N = float(np.median(df_nonbank['leverage']))
F_base_b, F_boot_b = recover_f_with_ci(struct_b, G_orig_b, G_boot_b,
                                         anchor_lev=ANCHOR_LEV_B)
F_base_n, F_boot_n = recover_f_with_ci(struct_n, G_orig_n, G_boot_n,
                                         anchor_lev=ANCHOR_LEV_N)
print(f'Bank f anchored at lev = {ANCHOR_LEV_B:.3f}')
print(f'Non-bank f anchored at lev = {ANCHOR_LEV_N:.3f}')

## Cell 11: Plotting function for $\hat f$ slices with bootstrap CIs

In [ ]:
def plot_f_slices_with_ci(struct, F_base, F_boot, port_std_data, anchor_lev,
                            quantiles=(0.25, 0.5, 0.75), label='', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(quantiles)))
    targets = np.quantile(port_std_data, quantiles)

    for q, target, color in zip(quantiles, targets, colors):
        j = np.argmin(np.abs(struct['X2'][:, 0] - target))
        x_axis = struct['X1'][j]
        mask_row = struct['mask'][j]
        base = np.where(mask_row, F_base[j], np.nan)
        boot = np.where(mask_row[None, :], F_boot[:, j, :], np.nan)
        lo = np.nanpercentile(boot, 2.5, axis=0)
        hi = np.nanpercentile(boot, 97.5, axis=0)
        ax.fill_between(x_axis, lo, hi, alpha=0.18, color=color)
        ax.plot(x_axis, base, color=color, lw=2,
                label=f'port_std = {target:.3f} ({int(q*100)}th)')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xlabel('Leverage  a/e')
    ax.set_ylabel(r'$\hat f$ (anchored at lev = ' + f'{anchor_lev:.2f}' + ')')
    ax.set_title(f'{label}: recovered $\\hat f$')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

## Cell 12: Combined figure for the report

2×2 layout. Top row: $\hat g$ slices for banks (left) and non-banks (right). Bottom row: recovered $\hat f$ slices. Saved to `nonparametric.png`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

plot_three_slices_with_ci(struct_b, G_orig_b, G_boot_b,
                            df_bank['port_std'].values,
                            label='Bank (two-way FE)', ax=axes[0, 0])
plot_three_slices_with_ci(struct_n, G_orig_n, G_boot_n,
                            df_nonbank['port_std'].values,
                            label='Non-Bank (two-way FE)', ax=axes[0, 1])

plot_f_slices_with_ci(struct_b, F_base_b, F_boot_b,
                       df_bank['port_std'].values, ANCHOR_LEV_B,
                       label='Bank (two-way FE)', ax=axes[1, 0])
plot_f_slices_with_ci(struct_n, F_base_n, F_boot_n,
                       df_nonbank['port_std'].values, ANCHOR_LEV_N,
                       label='Non-Bank (two-way FE)', ax=axes[1, 1])

plt.tight_layout()
plt.savefig('nonparametric.png', bbox_inches='tight', dpi=200)
plt.show()